# PneumoShift: Reproducible In-Domain Training (DenseNet121)

This standalone notebook retrains the in-domain baseline model on **NIH ChestX-ray14** and evaluates cross-dataset generalization on **Stanford CheXpert**.

### Guarantees & Protocol Compliance:
- **Architecture**: `densenet121` (matching Pan et al. 2019 / baseline specification)
- **Deterministic Partitioning**: Verified patient-level zero-leakage splits ($P_{train} \cap P_{val} = \emptyset$, $P_{train} \cap P_{test} = \emptyset$)
- **Zero Target Leakage**: Decision thresholds are strictly optimized on the source validation cohort
- **Atomic Resumability**: Checkpoints (`last.pt`) and run manifests are saved every epoch via write-to-temp-then-rename
- **Bounded Execution**: Bounded by `--max_hours` for safe Kaggle session management
- **Complete Array Persistence**: Output `.npz` files contain logits, probabilities, ground-truth labels, patient IDs, image IDs, pneumonia index, and penultimate representations

## 1. Global Session Configuration
Set the single random seed for this session (`42`, `1337`, or `2024`). Each Kaggle session executes **ONE** seed to remain comfortably within GPU quota.

In [ ]:
# ==============================================================================
# SESSION HYPERPARAMETERS & REPRODUCIBILITY CONTROLS
# ==============================================================================
ARCHITECTURE = "densenet121"
SEED = 42  # Options: 42, 1337, 2024

# Recipe Hyperparameters (must remain identical to baseline Pan et al. 2019)
EPOCHS = 10
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
IMAGE_SIZE = 224
MAX_HOURS = 11.5  # Safe exit prior to Kaggle 12-hour session limit

# Repository release pin (Pinned commit/tag, NEVER unpinned main branch)
GIT_REPO = "https://github.com/aditya-sharma2OO5/PneumoShift.git"
PINNED_COMMIT = "d867bfe"  # Pinned commit hash or release tag

# Optional uint8 resized preprocessing cache (OFF by default for bit-for-bit exactness)
ENABLE_UINT8_CACHE = False
print(f"Configured {ARCHITECTURE} training session for SEED={SEED}.")


## 2. Environment Setup & Pinned Repository Clone
Clone the exact repository commit and configure the working directory.

In [ ]:
import os
import sys
from pathlib import Path

# Clone repository if not present
repo_dir = Path("/kaggle/working/PneumoShift")
if not repo_dir.is_dir():
    !git clone {GIT_REPO} /kaggle/working/PneumoShift
    %cd /kaggle/working/PneumoShift
    !git checkout {PINNED_COMMIT}
else:
    %cd /kaggle/working/PneumoShift
    !git checkout {PINNED_COMMIT}

print("\nActive Git Commit:")
!git rev-parse HEAD
!git status --short

# Add repository root and src to sys.path
if str(repo_dir / 'src') not in sys.path:
    sys.path.insert(0, str(repo_dir / 'src'))


## 3. Dataset Linking & Path Resolution
Symlink NIH ChestX-ray14 and CheXpert images from Kaggle input datasets to `/tmp/nih_images`.

In [ ]:
import os
from pathlib import Path

# Create flat target directory for fast image access
nih_tmp = Path("/tmp/nih_images")
nih_tmp.mkdir(parents=True, exist_ok=True)

# Discover NIH image directories in /kaggle/input
candidate_dirs = [
    Path("/kaggle/input/data-nih/images"),
    Path("/kaggle/input/data-nih"),
    Path("/kaggle/input/nih-chest-xrays/images"),
    Path("/kaggle/input/nih-chest-xrays"),
    Path("/kaggle/input/data"),
]

found_images = 0
for cdir in candidate_dirs:
    if cdir.is_dir():
        for img in cdir.glob('*.png'):
            dest = nih_tmp / img.name
            if not dest.exists():
                try:
                    os.symlink(img, dest)
                    found_images += 1
                except Exception:
                    pass

print(f"Resolved and symlinked {found_images} NIH images into {nih_tmp}.")


## 4. Split Determinism & Checksum Verification
Assert that on-disk NIH split CSVs match the cryptographic hashes recorded in `data/splits/nih/nih_manifest.json` bit-for-bit.

In [ ]:
import hashlib
from pathlib import Path

EXPECTED_NIH_HASHES = {
    "nih_train.csv": "72749d41fadeff9a90fa6b69d09720ffd1855d2de49be4cf92b2930b06c477e7",
    "nih_val.csv": "79f974b2e0c20ba70cd324d6f8da52542a046855e10e10668b94b2a6d0ab3f2f",
    "nih_test.csv": "c0b2723519ab2a2c6cbbdcbb0369e55af9a69abcb1ebe41bcd4b1946a1e6e527",
    "nih_target_cal.csv": "973e3a3412d84c3a3f651ae7eb2db754b6642c3a01cbeb958cf901b826d2452d",
}

splits_dir = Path("data/splits/nih")
for filename, expected_hash in EXPECTED_NIH_HASHES.items():
    fpath = splits_dir / filename
    assert fpath.is_file(), f"ERROR: Missing required split file {fpath}"
    h = hashlib.sha256()
    with open(fpath, "rb") as fp:
        while chunk := fp.read(65536):
            h.update(chunk)
    actual_hash = h.hexdigest()
    assert actual_hash == expected_hash, (
        f"CRITICAL: SHA-256 hash mismatch for {filename}!\n"
        f"  Expected: {expected_hash}\n"
        f"  Actual:   {actual_hash}"
    )
    print(f"✓ {filename:20s}: {actual_hash[:20]}... [VERIFIED MATCH]")

print("\nAll NIH split checksums matched manifest bit-for-bit.")


## 5. Execution Smoke Test & Timing Projection
Run a 1-epoch smoke test to verify CUDA initialization, mixed-precision scaler, and compute projected duration for 10 epochs.

In [ ]:
!python scripts/train.py \
    --config configs/train/nih_densenet121.yaml \
    --seed {SEED} \
    --smoke_test


## 6. Full Model Training (10 Epochs, Resumable)
Execute the complete training pipeline:
- Monitored metric: `val_pneumonia_auroc`
- Optimizer: `AdamW (lr=1e-4, weight_decay=1e-5)`
- Scheduler: `ReduceLROnPlateau (factor=0.5, patience=2)`
- Early stopping: `patience=3`
- Evaluates best checkpoint on test set after training completes (`--eval_test`)

In [ ]:
!python scripts/train.py \
    --config configs/train/nih_densenet121.yaml \
    --seed {SEED} \
    --epochs {EPOCHS} \
    --eval_test \
    --resume \
    --max_hours {MAX_HOURS}


## 7. Cross-Domain Generalization Evaluation on Stanford CheXpert
Evaluate the best model checkpoint on the external Stanford CheXpert test cohort, extracting raw predictions, calibration metrics, and threshold transfer results.

In [ ]:
best_checkpoint = Path(f"results/runs/nih_{ARCHITECTURE}/seed_{SEED}/checkpoint_best.pt")
if not best_checkpoint.is_file():
    best_checkpoint = Path(f"results/runs/nih_{ARCHITECTURE}/checkpoints/checkpoint_best.pt")

print(f"Evaluating cross-domain performance using checkpoint: {best_checkpoint}")
!python scripts/evaluate_cross_domain.py \
    --config configs/eval/nih_to_chexpert.yaml \
    --checkpoint {str(best_checkpoint)} \
    --seed {SEED}


## 8. Run Manifest & Output Verification
Inspect the generated run manifest, completion marker, and verify that prediction arrays contain all required keys.

In [ ]:
import json
import numpy as np
from pathlib import Path

manifest_file = Path(f"results/runs/nih_{ARCHITECTURE}/seed_{SEED}/run_manifest_{ARCHITECTURE}_{SEED}.json")
if manifest_file.is_file():
    print("=== RUN MANIFEST ===")
    with open(manifest_file, "r") as fp:
        m = json.load(fp)
    for key in ["run_id", "architecture", "seed", "epochs", "best_epoch", "best_val_pneumonia_auroc", "gpu_name", "checkpoint_sha256"]:
        print(f"  {key:25s}: {m.get(key)}")

print("\n=== AUDITING SAVED PREDICTION ARRAYS (.npz) ===")
pred_dir = Path("results/predictions")
for npz_file in pred_dir.glob(f"*{ARCHITECTURE}_{SEED}*.npz"):
    data = np.load(npz_file, allow_pickle=True)
    print(f"{npz_file.name:45s}: keys={list(data.files)}")
    assert "image_ids" in data.files, f"Missing image_ids in {npz_file.name}"
    assert "patient_ids" in data.files, f"Missing patient_ids in {npz_file.name}"
    assert "y_true" in data.files, f"Missing y_true in {npz_file.name}"
    assert "y_score" in data.files, f"Missing y_score in {npz_file.name}"
    assert "pneumonia_col_idx" in data.files, f"Missing pneumonia_col_idx in {npz_file.name}"

print("\nAll prediction outputs verified successfully with full key coverage.")


## 9. Package Artifacts into Downloadable Zip
Bundle checkpoints, predictions, logs, manifests, and table summaries into a single archive for preservation.

In [ ]:
import shutil
from pathlib import Path

bundle_name = f"pneumoshift_artifacts_{ARCHITECTURE}_seed_{SEED}"
export_root = Path(f"/kaggle/working/{bundle_name}")
export_root.mkdir(parents=True, exist_ok=True)

# Copy seed runs folder
run_dir = Path(f"results/runs/nih_{ARCHITECTURE}/seed_{SEED}")
if run_dir.is_dir():
    shutil.copytree(run_dir, export_root / "runs" / f"seed_{SEED}", dirs_exist_ok=True)

# Copy prediction arrays
pred_dir = Path("results/predictions")
export_pred = export_root / "predictions"
export_pred.mkdir(parents=True, exist_ok=True)
for f in pred_dir.glob(f"*{ARCHITECTURE}_{SEED}*"):
    shutil.copy2(f, export_pred / f.name)

# Copy generated tables and reports
tables_src = Path(f"results/tables/cross_domain/nih_to_chexpert")
if tables_src.is_dir():
    shutil.copytree(tables_src, export_root / "tables", dirs_exist_ok=True)

zip_output = f"/kaggle/working/{bundle_name}"
shutil.make_archive(zip_output, "zip", export_root)
print(f"Artifact bundle created: {zip_output}.zip")
!ls -lh /kaggle/working/*.zip


## 10. Optional: Pre-Resized uint8 Image Cache (Disabled by Default)
If training I/O from disk is a bottleneck, the cell below builds an uncompressed uint8 image cache (`224x224`) once before training.
Because this applies exact bilinear resizing to 224x224 prior to random augmentations, it preserves bit-for-bit fidelity with PIL decoding while accelerating DataLoader throughput.

In [ ]:
# This cell is OFF by default. Set ENABLE_UINT8_CACHE = True in Cell 1 to activate.
if ENABLE_UINT8_CACHE:
    from PIL import Image
    from tqdm import tqdm
    cache_dir = Path("/tmp/nih_cache_224")
    cache_dir.mkdir(parents=True, exist_ok=True)
    img_files = list(Path("/tmp/nih_images").glob("*.png"))
    print(f"Pre-caching {len(img_files)} images to {cache_dir}...")
    for p in tqdm(img_files):
        dst = cache_dir / p.name
        if not dst.exists():
            im = Image.open(p).convert("RGB").resize((224, 224), Image.BILINEAR)
            im.save(dst)
    print("Pre-caching completed.")
else:
    print("uint8 image cache is DISABLED (using standard on-the-fly transforms).")
